# 02 — Data description and exploratory overview

This notebook describes the structured outputs created by `01_collection_inventory.ipynb`.
It does **not** rescan the PDF files. Instead, it reads the inventory tables under `../output/inventory` and summarizes the collection's structure, completeness, metadata coverage, document characteristics, and supplied terms.


## 1. Dependencies and paths

Run this notebook from the `notebooks/` folder, same as `01_collection_inventory.ipynb`.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')

INVENTORY_DIR = Path('../output/inventory').resolve()

if not INVENTORY_DIR.is_dir():
    raise FileNotFoundError(
        f'Inventory folder not found: {INVENTORY_DIR}. '
        'Run 01_collection_inventory.ipynb first.'
    )

print(f'Inventory folder: {INVENTORY_DIR}')


## 2. Load the inventory tables

The collection inventory contains three related tables:

- `documents.csv`: one row per unique PDF content.
- `metadata_observations.csv`: observed metadata candidates and their sources.
- `document_terms.csv`: embedded or manually supplied keywords, themes, or authors.

`document_id` is used as the common identifier between the tables.

In [ ]:
def load_csv(filename):
    path = INVENTORY_DIR / filename
    if not path.exists():
        raise FileNotFoundError(f'Missing required file: {path}')
    return pd.read_csv(path, dtype=str, keep_default_na=False, encoding='utf-8-sig')

documents = load_csv('documents.csv')
observations = load_csv('metadata_observations.csv')
terms = load_csv('document_terms.csv')

print(f'Documents: {len(documents):,} rows')
print(f'Metadata observations: {len(observations):,} rows')
print(f'Document terms: {len(terms):,} rows')


## 3. Dataset structure

Inspect the dimensions and columns of each table. This provides a basic description of what data is available before analysing individual fields.

In [ ]:
for name, df in [('documents', documents), ('observations', observations), ('terms', terms)]:
    print(f'\n{name.upper()}')
    print(f'Rows: {len(df):,}')
    print(f'Columns: {len(df.columns):,}')
    display(pd.DataFrame({'column': df.columns, 'dtype': [str(df[c].dtype) for c in df.columns]}))


## 4. Document-level overview

The `documents` table is the main unit of analysis. The following summaries describe the number of PDFs, page counts, extracted text volume, and file sizes.

In [ ]:
numeric_columns = ['file_size_bytes', 'page_count', 'extracted_char_count', 'pages_without_text']
for col in numeric_columns:
    documents[col] = pd.to_numeric(documents[col], errors='coerce')

summary = pd.DataFrame({
    'Measure': [
        'Unique documents',
        'Total pages',
        'Total extracted characters',
        'Average pages per document',
        'Median pages per document',
        'Average extracted characters per document',
        'Median extracted characters per document',
        'Total file size (MB)'
    ],
    'Value': [
        len(documents),
        documents['page_count'].sum(),
        documents['extracted_char_count'].sum(),
        documents['page_count'].mean(),
        documents['page_count'].median(),
        documents['extracted_char_count'].mean(),
        documents['extracted_char_count'].median(),
        documents['file_size_bytes'].sum() / 1000000
    ]
})
display(summary)

display(documents[['file_size_bytes', 'page_count', 'extracted_char_count', 'pages_without_text']].describe().T)


## 5. Text extraction quality

`text_status` was generated during PDF extraction. It distinguishes fully extractable PDFs from PDFs with partial or absent text and from extraction errors.

This is useful for identifying documents that may require OCR or manual inspection later.

In [ ]:
text_status = (documents['text_status']
               .value_counts(dropna=False)
               .rename_axis('text_status')
               .to_frame('documents'))
text_status['percentage'] = text_status['documents'] / len(documents) * 100 if len(documents) else 0
display(text_status)

if len(documents):
    ax = documents['text_status'].value_counts().plot(kind='bar')
    ax.set_title('PDFs by text extraction status')
    ax.set_xlabel('Text status')
    ax.set_ylabel('Number of PDFs')
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.show()


## 6. Document size and page-count distributions

These distributions show whether the collection consists mainly of short documents or whether a smaller number of long documents contribute a large amount of text.

In [ ]:
fig, ax = plt.subplots()
documents['page_count'].dropna().plot(kind='hist', bins=30, ax=ax)
ax.set_title('Distribution of document page counts')
ax.set_xlabel('Pages')
ax.set_ylabel('Number of PDFs')
plt.tight_layout()
plt.show()

fig, ax = plt.subplots()
documents['extracted_char_count'].dropna().plot(kind='hist', bins=30, ax=ax)
ax.set_title('Distribution of extracted text length')
ax.set_xlabel('Extracted characters')
ax.set_ylabel('Number of PDFs')
plt.tight_layout()
plt.show()


## 7. Metadata coverage

The inventory separates document-level reviewed fields from raw observations. Here we measure how many documents have at least one observation for each field.

This describes coverage, not correctness. An automatically extracted DOI may be wrong.

In [ ]:
if observations.empty:
    print('No metadata observations are available.')
else:
    coverage = (observations.groupby('field_name')['document_id']
                .nunique()
                .sort_values(ascending=False)
                .rename('documents_with_observation')
                .to_frame())
    coverage['coverage_percentage'] = coverage['documents_with_observation'] / len(documents) * 100 if len(documents) else 0
    display(coverage)

    ax = coverage['coverage_percentage'].sort_values().plot(kind='barh')
    ax.set_title('Metadata observation coverage')
    ax.set_xlabel('Documents with observation (%)')
    ax.set_ylabel('Metadata field')
    plt.tight_layout()
    plt.show()


## 8. Metadata review status

The `documents.csv` fields selected for human review can be summarized by their current review status.

In [ ]:
if 'metadata_review_status' in documents.columns:
    review_status = (documents['metadata_review_status']
                    .replace('', 'unknown/unset')
                    .value_counts(dropna=False)
                    .rename_axis('metadata_review_status')
                    .to_frame('documents'))
    review_status['percentage'] = review_status['documents'] / len(documents) * 100 if len(documents) else 0
    display(review_status)

review_fields = [
    'title', 'doi', 'publication_year', 'publication_date', 'online_date',
    'document_type', 'language'
]

available_review_fields = [c for c in review_fields if c in documents.columns]
if available_review_fields:
    missingness = pd.DataFrame({
        'field': available_review_fields,
        'filled_documents': [documents[c].replace('', pd.NA).notna().sum() for c in available_review_fields]
    })
    missingness['missing_documents'] = len(documents) - missingness['filled_documents']
    missingness['filled_percentage'] = missingness['filled_documents'] / len(documents) * 100 if len(documents) else 0
    display(missingness)


## 9. Publication years and document types

When these fields have been manually reviewed, summarize their distributions. Empty values are excluded rather than interpreted as a particular year or type.

In [ ]:
if 'publication_year' in documents.columns:
    years = pd.to_numeric(documents['publication_year'], errors='coerce').dropna()
    if not years.empty:
        print(f'Publication year range: {int(years.min())}–{int(years.max())}')
        display(years.astype(int).value_counts().sort_index().rename('documents').to_frame())

        fig, ax = plt.subplots()
        years.astype(int).value_counts().sort_index().plot(kind='bar', ax=ax)
        ax.set_title('Documents by publication year')
        ax.set_xlabel('Publication year')
        ax.set_ylabel('Number of documents')
        plt.tight_layout()
        plt.show()
    else:
        print('No reviewed publication years are available yet.')

if 'document_type' in documents.columns:
    types = documents['document_type'].replace('', pd.NA).dropna()
    if not types.empty:
        display(types.value_counts().rename('documents').to_frame())
    else:
        print('No reviewed document types are available yet.')

if 'language' in documents.columns:
    languages = documents['language'].replace('', pd.NA).dropna()
    if not languages.empty:
        display(languages.value_counts().rename('documents').to_frame())
    else:
        print('No reviewed languages are available yet.')


## 10. Source and extraction methods

The observations table records where metadata came from and how it was obtained. Important when assessing the reliability and reproducibility of later metadata processing.

In [ ]:
if observations.empty:
    print('No observations available.')
else:
    print('Observation source types:')
    display(observations['source_type'].value_counts(dropna=False).rename('observations').to_frame())

    print('Extraction methods:')
    display(observations['extraction_method'].value_counts(dropna=False).rename('observations').to_frame())

    print('Observation review status:')
    display(observations['review_status'].replace('', 'unknown/unset')
            .value_counts(dropna=False).rename('observations').to_frame())


## 11. Document terms

`document_terms.csv` can contain embedded keywords as well as manually reviewed terms. The summaries below describe how many documents have terms and which term types are present.

In [ ]:
if terms.empty:
    print('No document terms are available.')
else:
    terms_per_document = terms.groupby('document_id').size()
    print(f'Documents with at least one term: {terms_per_document.index.nunique():,}')
    print(f'Average terms per document among documents with terms: {terms_per_document.mean():.2f}')
    print(f'Median terms per document among documents with terms: {terms_per_document.median():.0f}')

    print('\nTerm types:')
    display(terms['term_type'].replace('', 'unknown/unset').value_counts().rename('terms').to_frame())

    print('\nTerm source types:')
    display(terms['source_type'].replace('', 'unknown/unset').value_counts().rename('terms').to_frame())

    print('\nMost frequent terms:')
    display(terms['term_value'].replace('', pd.NA).dropna().str.lower()
            .value_counts().head(20).rename('documents/occurrences').to_frame())
